# 10. 프로덕션 — 테스트, 배포, 관측성

## 학습 목표

LangGraph 앱을 테스트, 배포, 모니터링하는 방법을 알아봅니다.

## 10.1 환경 설정

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-4.1")

In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


Langfuse tracing ON — https://lf.ddok.ai


## 10.2 앱 구조 — langgraph.json

- `langgraph.json`: 그래프 정의, 의존성, 환경 변수 설정
- `langgraph dev`: 로컬 개발 서버 실행

In [3]:
import json

config = {
    "dependencies": ["."],
    "graphs": {
        "agent": "./agent.py:graph"
    },
    "env": ".env"
}
print("langgraph.json 예시:")
print(json.dumps(config, indent=2))
print()
print("명령어:")
print("  $ pip install 'langgraph-cli[inmem]'")
print("  $ langgraph dev  # http://localhost:2024에서 로컬 서버 시작")

langgraph.json 예시:
{
  "dependencies": [
    "."
  ],
  "graphs": {
    "agent": "./agent.py:graph"
  },
  "env": ".env"
}

명령어:
  $ pip install 'langgraph-cli[inmem]'
  $ langgraph dev  # http://localhost:2024에서 로컬 서버 시작


## 10.3 LangGraph Studio — 시각적 디버깅 도구

Studio는 `langgraph dev` 실행 시 자동으로 제공됩니다.

**기능:**
- 그래프 구조 시각화
- 실시간 실행 추적
- 상태 검사 및 수정
- 인터랙티브 테스트
- 체크포인트 탐색 (타임 트래블)

**사용 방법:**
```bash
$ langgraph dev
# 브라우저에서 http://localhost:2024 접속
# 또는 LangSmith Studio에서 원격 접속
```

## 10.4 Agent Chat UI

채팅 인터페이스로 에이전트와 대화:

```bash
$ npx @anthropic-ai/agent-chat-ui
```

**기능:**
- 실시간 스트리밍 채팅
- 도구 호출 시각화
- 대화 분기 (branching)
- Human-in-the-loop 승인
- 멀티 에이전트 메시지 구분

## 10.5 테스트 — 결정론적 에이전트 테스트

In [4]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

# Graph to test
class TestState(TypedDict):
    input: str
    output: str


def process(state: TestState) -> dict:
    return {"output": state["input"].upper()}


builder = StateGraph(TestState)

builder.add_node("process", process)
builder.add_edge(START, "process")
builder.add_edge("process", END)

graph = builder.compile()


# Unit tests
def test_process():
    result = graph.invoke({"input": "hello"})

    assert result["output"] == "HELLO", f"HELLO 예상, {result['output']} 반환됨"

    print("  OK test_process")


def test_empty_input():
    result = graph.invoke({"input": ""})

    assert result["output"] == "", f"빈 문자열 예상, {result['output']} 반환됨"

    print("  OK test_empty_input")


print("테스트 실행 중:")

test_process()
test_empty_input()

print("모든 테스트 통과!")

테스트 실행 중:
  OK test_process
  OK test_empty_input
모든 테스트 통과!


## 10.6 LLM 에이전트 테스트 — GenericFakeChatModel 사용

In [5]:
from langchain_core.language_models import GenericFakeChatModel
from langchain.messages import AIMessage, HumanMessage, AnyMessage
from langgraph.graph import StateGraph, START, END, MessagesState

# Deterministic fake model
fake_model = GenericFakeChatModel(
    messages=iter(
        [
            AIMessage(content="The answer is 42."),
        ]
    )
)

def chatbot(state: MessagesState) -> dict:
    return {
        "messages": [fake_model.invoke(state["messages"])]
    }

builder = StateGraph(MessagesState)

builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

test_graph = builder.compile()

result = test_graph.invoke(
    {
        "messages": [HumanMessage(content="테스트")]
    }
)

assert "42" in result["messages"][-1].content

print("GenericFakeChatModel 테스트 통과!")
print(f"  응답: {result['messages'][-1].content}")

GenericFakeChatModel 테스트 통과!
  응답: The answer is 42.


## 10.7 배포 옵션

**1. LangGraph Platform (managed):**
```bash
$ langgraph deploy
```

**2. Self-hosted Docker:**
```bash
$ langgraph build -t my-agent
$ docker run -p 2024:2024 my-agent
```

**3. LangGraph Cloud:**
- GitHub 연동 자동 배포
- https://smith.langchain.com 에서 관리

## 10.8 관측성 — LangSmith 트레이싱

**설정 (`.env`):**
```
LANGSMITH_API_KEY=lsv2-...
LANGSMITH_TRACING=true
```

**자동 추적 항목:**
- 각 노드 실행 시간
- LLM 입출력, 토큰 사용량
- 도구 호출 및 결과
- 상태 변화
- 에러 및 재시도

In [6]:
import os

tracing = os.environ.get("LANGSMITH_TRACING", "false")
api_key = os.environ.get("LANGSMITH_API_KEY", "")

print("현재 상태:")

print(f"  트레이싱: {'활성화' if tracing == 'true' else '비활성화'}")
print(f"  API 키: {'설정됨' if api_key else '미설정'}")

현재 상태:
  트레이싱: 비활성화
  API 키: 미설정


## 10.9 Pregel 런타임 개요

- **Pregel**은 LangGraph의 내부 실행 엔진
- Graph API와 Functional API 모두 Pregel 위에서 실행됨
- 핵심 개념: **슈퍼스텝**, **채널**, **체크포인트**
- **슈퍼스텝**: 동일 레벨의 노드가 병렬 실행되는 단위
- 일반적으로 직접 사용할 필요 없음 (Graph/Functional API가 추상화)

**LangGraph 실행 모델:**

```
[Super-step 1] Node A, Node B (병렬)
     ↓ 상태 업데이트
[Super-step 2] Node C (A, B 결과 기반)
     ↓ 상태 업데이트
[Super-step 3] Node D
     ↓
END
```

**각 슈퍼스텝:**
1. 해당 노드들 병렬 실행
2. 상태 업데이트 (리듀서 적용)
3. 체크포인트 저장
4. 다음 슈퍼스텝 결정

## 10.10 프로덕션 체크리스트

| 항목 | 도구 | 설명 |
|------|------|------|
| 단위 테스트 | pytest | 개별 노드 함수 테스트 |
| 통합 테스트 | GenericFakeChatModel | API 호출 없이 전체 흐름 |
| 지속성 | PostgresSaver | 프로덕션 체크포인터 |
| 관측성 | LangSmith | 트레이싱, 모니터링 |
| 배포 | langgraph deploy | 관리형 배포 |
| UI | Agent Chat UI | 사용자 인터페이스 |

## 10.11 요약

| 주제 | 핵심 내용 |
|------|----------|
| 앱 구조 | `langgraph.json`으로 프로젝트 설정 |
| Studio | `langgraph dev`로 시각적 디버깅 |
| 테스트 | 결정론적 테스트 + GenericFakeChatModel |
| 배포 | Platform, Docker, Cloud 옵션 |
| 관측성 | LangSmith 트레이싱 |
| 런타임 | Pregel 슈퍼스텝 실행 모델 → [13번 노트북](13_api_guide_and_pregel.ipynb)에서 심화 |

### 다음 단계
→ **[11. 로컬 서버](11_local_server.ipynb)**으로 진행하세요!
→ **[Deep Agents 중급 과정](../04_deepagents/01_introduction.ipynb)**으로 건너뛰기